# MNAI Nano 2 — 0.9B from scratch

Run this in Colab with a GPU runtime (T4, free). It clones the repo, trains the tokenizer, trains the model with resume, then makes a GGUF.

Made by MeowSamOPDZN Studio. Owner: MeowSamOP DZN.

## 1. Setup — clone, install, check GPU

In [ ]:
!nvidia-smi
!git clone https://github.com/MeowSamOPDZN/mnai-nano-2.git
%cd mnai-nano-2
!pip install -q -r requirements.txt
!python model.py

## 2. Save work to Drive (so a session ending does not lose checkpoints)

Mount Drive, then training writes under Drive. Skip this if you are just smoke-testing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/mnai-nano-2
!ln -sfn /content/drive/MyDrive/mnai-nano-2 /content/mnai-nano-2/out-09b
!ls /content/drive/MyDrive/mnai-nano-2

## 3. Train the tokenizer (once, a few minutes)

In [ ]:
!python train_tokenizer.py --out tokenizer.json --num-docs 50000

## 4. Train — first run

1B tokens is the first useful target. This will NOT finish in one Colab session. That is normal, just resume in the next cell.

In [ ]:
!python train.py --tokenizer tokenizer.json --out out-09b --max-tokens 1000000000 --micro-batch 4 --grad-accum 16

## 5. Resume — run this instead of step 4 after a disconnect

Change ckpt-500 to your latest checkpoint folder.

In [ ]:
!ls out-09b
!python train.py --tokenizer tokenizer.json --out out-09b --resume out-09b/ckpt-500 --max-tokens 1000000000

## 6. Test it

In [ ]:
!python generate.py --model out-09b/ckpt-500 --prompt "MNAI was created by" --max-new 120

## 7. Make the GGUF

Q4_K_M is about 550MB and runs on a 4GB RAM laptop in LM Studio. Download the .gguf file from the file browser on the left when it is done.

In [ ]:
!python convert_to_gguf.py --model out-09b/ckpt-500 --out mnai-nano-2-q4_k_m.gguf
!ls -lh *.gguf